# Exercise 4: E-Commerce Text & Quantitative Analytics

Objective: Perform group-level statistical aggregation and unstructured text visualization
on real-world customer review data.
* Dataset: Flipkart Product Customer Reviews Dataset (Kaggle)

Task Description:
1. Load the dataset into a Pandas DataFrame and perform any necessary data
cleaning on the rating and price fields.
2. Group the records by product name, calculate the mean rating and mean price
for each product, and store the resulting summary table in a new DataFrame
named items.
3. Using the text data in the Summary column, generate and

### Import Required Libraries

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from wordcloud import WordCloud, STOPWORDS

%matplotlib inline

### Task 1: Load dataset & Perform data cleaning
**Data Quality Findings & Fixes for `flipkart_product.csv`:**
- **File Encoding:** `flipkart_product.csv` contains non-UTF-8 bytes (`0xff`), so it is loaded with `encoding='latin1'`.
- **Price Formatting:** Currency symbols/mojibake and commas (e.g., `ƒ??3,999`) are stripped before converting to `float`.
- **Corrupted Rating Rows:** A few shifted rows contain product titles (such as `'Pigeon Favourite Electric Kettle (1.5 L...)'`) or `'s'` inside `Rate`. Using strict `pd.to_numeric()` after stripping whitespace ensures corrupted rows become `NaN` rather than mistakenly extracting numbers like `1.5` from product titles.

In [ ]:
# Load flipkart_product.csv using latin1 encoding
csv_path = 'flipkart_product.csv'
df = pd.read_csv(csv_path, encoding='latin1')
print(f"Raw dataset loaded: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Standardize column names
df = df.rename(columns={
    'ProductName': 'product_name',
    'Price': 'price',
    'Rate': 'rating'
})

# 1. Clean 'price': strip currency symbols, commas, and non-numeric characters
df['price'] = (
    df['price']
    .astype(str)
    .str.replace(r'[^0-9.]', '', regex=True)
    .replace('', np.nan)
)
df['price'] = pd.to_numeric(df['price'], errors='coerce')

# 2. Clean 'rating': parse pure numeric ratings and enforce valid 1-5 range
df['rating'] = pd.to_numeric(df['rating'].astype(str).str.strip(), errors='coerce')
df.loc[(df['rating'] < 1) | (df['rating'] > 5), 'rating'] = np.nan

# 3. Drop rows with missing/corrupted product_name, rating, or price
df_clean = df.dropna(subset=['product_name', 'rating', 'price']).copy()

# Clean non-breaking space / encoding artifacts from product_name
df_clean['product_name'] = (
    df_clean['product_name']
    .astype(str)
    .str.replace(r'[\xa0Â\?ÿ]+', ' ', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

print(f"Valid rows after cleaning: {len(df_clean):,} (dropped {len(df) - len(df_clean)} corrupted/incomplete rows)")
display(df_clean[['product_name', 'price', 'rating', 'Summary']].head())

### Task 2: Group by Product Name & Compute Mean Rating and Mean Price (`items`)

In [ ]:
# Group by product_name and compute mean rating and mean price
items = (
    df_clean
    .groupby('product_name', as_index=False)
    .agg(
        mean_rating=('rating', 'mean'),
        mean_price=('price', 'mean'),
        review_count=('rating', 'count')
    )
    .sort_values(by=['review_count', 'mean_rating'], ascending=[False, False])
    .reset_index(drop=True)
)

items['mean_rating'] = items['mean_rating'].round(2)
items['mean_price'] = items['mean_price'].round(2)

print(f"Total unique products in 'items': {len(items):,}")
display(items.head(15))

### Task 3: Generate & Display Word Cloud from the `Summary` Column

In [ ]:
# Combine all non-null feedback text from the 'Summary' column
raw_text = ' '.join(df_clean['Summary'].dropna().astype(str).tolist())

# Keep alphabetic words only (strips punctuation, numbers, and encoding symbols)
cleaned_text = re.sub(r'[^A-Za-z\s]', ' ', raw_text).lower()

# Configure stopwords
custom_stopwords = set(STOPWORDS)
custom_stopwords.update(['product', 'flipkart', 'buy', 'bought', 'one', 'use', 'using'])

# Generate WordCloud
wordcloud = WordCloud(
    width=1200,
    height=600,
    background_color='white',
    colormap='viridis',
    stopwords=custom_stopwords,
    max_words=150,
    collocations=False
).generate(cleaned_text)

# Display and save the WordCloud
plt.figure(figsize=(14, 7))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Most Frequent Customer Feedback Terms (Summary Column)', fontsize=16, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('flipkart_summary_wordcloud.png', dpi=150, bbox_inches='tight')
plt.show()